# 07b. 강수량 시나리오 예측

## 목적
"같은 날에 비가 안 왔다면 vs 45mm 왔다면 매출이 얼마나 달라질까?"를 예측 모델로 계산한다.
업종·지역별로 호우에 얼마나 민감한지 **위험 순위**를 매기는 데 사용한다.

## 06 분석과의 역할 차이
- 06 회귀분석: 과거에 호우일 매출이 실제로 **얼마나** 줄었는지 (피해 크기)
- 07b 시나리오: 비가 오면 **어느 업종이 가장 위험한지** 미리 알려주는 도구 (위험 순위)

## 1. 준비
- 구글 드라이브 연결
- 한글 폰트(나눔고딕) 설치: 코랩 기본 폰트에는 한글이 없어 그래프 글자가 깨지기 때문
- `axes.unicode_minus = False`: 그래프의 마이너스(−) 기호 깨짐 방지
- 경로 지정: 입력은 `data/processed`, 결과 표는 `outputs/tables`, 그래프는 `outputs/figures`


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq install -y fonts-nanum > /dev/null
!pip install -q lightgbm

from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data" / "processed"
TAB = BASE / "outputs" / "tables"
FIG = BASE / "outputs" / "figures"
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

## 2. 모델 학습 + 시나리오 계산

### 모델
- 입력: `model_input.csv` (날짜 × 지역 × 업종, 4,602행)
- 07의 ③ 날씨 포함 LightGBM과 **같은 변수·같은 설정**으로 전체 데이터를 학습
  - 과거 매출·달력 변수 10개(주 번호 포함) + 날씨 변수 9개
  - 설정: 나무 500그루, 학습률 0.03, 잎 31개, 깊이 제한 없음 (07 ③과 동일)
  - 정답(y) = log(매출) → 예측값에 exp를 씌워 원래 금액으로 되돌림

### 시나리오 방법
1. 실제로 **비가 안 온 여름날(7~9월, 강수 1mm 미만)**만 고른다
   → 원래 맑은 날이라 "비만 바뀐 효과"를 깨끗하게 볼 수 있음
2. 그 날들의 다른 조건(요일, 기온, 지난주 매출 등)은 그대로 두고 **강수량만** 0 / 10 / 30 / 45mm로 바꿔서 예측한다
   - 강수 여부(rainy): 1mm 이상이면 1
   - 호우 여부(heavy): 30mm 이상이면 1
3. 지역·업종별로 `(비 온 경우 예측 합계 ÷ 0mm 예측 합계 − 1) × 100` = 예상 매출 변화(%)

### 기준값 근거
- 10mm: 호우 주의 수준
- 30mm: 호우 경보 수준 (분석에서 쓴 호우 기준)
- 45mm: 실제 춘천 호우일 수준의 큰 비

### 결과 저장
`outputs/tables/07_scenario.csv`

In [ ]:
df = pd.read_csv(OUT / "model_input.csv", parse_dates=["date"])
for c in ["region", "ind_grp"]:
    df[c] = df[c].astype("category")

# 07 성능표의 ③ 날씨 포함 모델과 같은 변수·설정
BASE_FEATS = ["week","dow","weekend","holiday","chuseok","month","lag7","lag7_mean4","region","ind_grp"]
WX_FEATS = ["tmax","feels_max","rain","rainy","heavy","heat","heat_streak","tmax_lag1","rain_lag1"]
FEATS = BASE_FEATS + WX_FEATS
PARAMS = dict(objective="regression", n_estimators=500, learning_rate=0.03, num_leaves=31,
              max_depth=-1, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)

final_wx = lgb.LGBMRegressor(**PARAMS).fit(df[FEATS], df["y"])

def predict_with_rain(d, mm):
    s = d.copy()
    s["rain"] = mm
    s["rainy"] = int(mm >= 1)
    s["heavy"] = int(mm >= 30)
    return np.exp(final_wx.predict(s[FEATS]))

base = df[(df["month"].between(7, 9)) & (df["rain"] < 1)]   # 실제로 비 안 온 여름날

rows = []
for (r, g), d in base.groupby(["region", "ind_grp"], observed=True):
    dry = predict_with_rain(d, 0).sum()
    row = {"지역": r, "업종": g}
    for mm in [10, 30, 45]:
        row[f"{mm}mm"] = round((predict_with_rain(d, mm).sum() / dry - 1) * 100, 1)
    rows.append(row)

scen = pd.DataFrame(rows).sort_values("45mm")
print(scen.to_string(index=False))
scen.to_csv(TAB / "07_scenario.csv", index=False, encoding="utf-8-sig")
print(base.groupby("region", observed=True)["date"].nunique())

## 3. 그래프: 45mm 호우 시 업종별 예상 매출 변화
- 빨강 = 춘천, 파랑 = 강남
- 막대가 왼쪽으로 길수록 비에 취약한 업종
- 춘천 대형유통은 06 분석에서 해석 제외(기타유통 주도)라 그래프에서 뺐다
- 저장: `outputs/figures/07_scenario_45mm.png`

In [ ]:
s = scen[~((scen["지역"] == "춘천") & (scen["업종"] == "대형유통"))]   # 06에서 해석 제외한 업종
p = s.pivot(index="업종", columns="지역", values="45mm").sort_values("춘천")

ax = p[["춘천", "강남"]].plot.barh(figsize=(8, 6), color=["#d1495b", "#8fb3d9"])
ax.set_xlabel("0mm 대비 예상 매출 변화 (%) — 45mm 호우")
ax.set_ylabel("")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("강수량 시나리오: 호우 시 업종별 예상 매출 변화")
plt.tight_layout()
plt.savefig(FIG / "07_scenario_45mm.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
show = [("춘천","야외레저"), ("강남","야외레저"), ("춘천","카페·제과"), ("강남","카페·제과")]
mm = [0, 10, 30, 45]

fig, ax = plt.subplots(figsize=(8, 5))
for (r, g) in show:
    row = scen[(scen["지역"] == r) & (scen["업종"] == g)].iloc[0]
    vals = [0, row["10mm"], row["30mm"], row["45mm"]]
    style = dict(color="#d1495b" if r == "춘천" else "#8fb3d9",
                 lw=3 if g == "야외레저" else 1.5,
                 ls="-" if g == "야외레저" else "--", marker="o")
    ax.plot(mm, vals, label=f"{r} {g}", **style)
    ax.annotate(f"{vals[-1]:.0f}%", (45, vals[-1]), xytext=(6, 0),
                textcoords="offset points", va="center")

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(mm)
ax.set_xlabel("강수량 (mm)")
ax.set_ylabel("비 안 온 날 대비 예상 매출 변화 (%)")
ax.set_title("비가 많이 올수록 춘천 야외레저만 크게 줄어든다")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "07_scenario_line.png", dpi=200, bbox_inches="tight")
plt.show()

## 결과 해석
- 45mm 호우 시 예상 매출 감소가 가장 큰 곳은 **춘천 야외레저(−34.6%)**로, 다른 업종(대부분 −10% 이내)보다 3배 이상 컸다.
  → 06 회귀분석에서 호우 피해가 가장 컸던 업종(−53.4%)과 같다.
- 춘천 야외레저는 강수량이 늘수록 감소폭이 커졌다 (10mm −5.4% → 30mm −24.7% → 45mm −34.6%).
  → 06 구간별 분석(1~10mm −12.5%, 10~30mm −30.5%, 30mm 이상 −56.2%)과 같은 패턴이다.
- 같은 야외레저라도 강남은 −9.8%로 춘천의 1/3 수준이었다.
- 나머지 업종은 대부분 ±5% 안팎으로 작고, 일부는 방향이 일정하지 않아(예: 강남 숙박 +3.9%) 순위를 해석하지 않았다.

## 한계
- 트리 모델은 표본이 적은 조합(춘천 야외레저 호우일 17일)의 효과를 **보수적으로** 추정한다 (모델 −34.6% vs 회귀분석 −53.4%).
- 따라서 **피해 크기는 06 회귀분석 값을, 시나리오는 위험 순위 판단에** 사용한다.
- 작은 효과(±5% 수준)는 모델 설정에 따라 값이 흔들리므로 해석하지 않는다.
- 과거 예보 자료가 없어 실측 강수량으로 대신했다. 실제 운영에서는 예보 강수량을 넣는다.